In [ ]:
from datasets import *
from network import *
from loss import *
from training import *

In [ ]:
H5_PATH = "training.h5"
BATCH_SIZE = 16
dataset = LArTPCSequenceDataset(H5_PATH)

In [ ]:
from torch.utils.data import Subset
dataset = Subset(dataset, range(3000))

from torch.utils.data import random_split

val_frac = 0.1
num_events = len(dataset)
num_val = int(num_events * val_frac)
num_train = num_events - num_val

generator = torch.Generator().manual_seed(42)
train_dataset, val_dataset = random_split(dataset, [num_train, num_val], generator=generator)

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, collate_fn=collate_fn, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, collate_fn=collate_fn, shuffle=False, num_workers=0)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
print(torch.version.cuda)

In [ ]:
model = ObjectCondensationNet(
    input_dim=3,          # xx, zz, vv
    embed_dim=128,
    num_heads=4,
    num_layers=4,
    ff_dim=256,
    num_classes=None,     # OC doesn't need this
    dropout=0.1
).to(device)

In [ ]:
criterion = ObjectCondensationLoss()

In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4,
)

In [ ]:
scaler = torch.amp.GradScaler('cuda')

In [ ]:
num_epochs = 3

## Training Loop

In [ ]:
best_val_loss = float("inf")

for epoch in range(1, num_epochs + 1):
    print(f"\n===== Epoch {epoch}/{num_epochs} =====")

    train_loss = train_one_epoch(
        model=model,
        train_loader=train_loader,
        optimizer=optimizer,
        criterion=criterion,
        device=device,
        scaler=scaler,
        max_grad_norm=1.0
    )

    torch.save(model.state_dict(), f"model_epoch_{epoch}.pt")

    val_loss = validate(
        model=model,
        val_loader=val_loader,
        criterion=criterion,
        device=device
    )

    print(f"Epoch {epoch}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}")

    # ---------------------------------
    # Save best model
    # ---------------------------------
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "best_oc_model.pt")
        print("  → Saved best model")